# Octonion-GPT on Kaggle - gradient-free, O(n), large-corpus generation

A gradient-free language generator built on the **octonion algebra / Fano plane** (no
backprop, no GPU). It combines, on one large corpus:

- **meaning** - PMI-SVD semantic embedding (Levy-Goldberg: word2vec without gradients)
- **fluency** - trigram backbone (counts)
- **topic flow** - an evolving discourse state
- **proposition** - an octonion **bind** state (`role (x) word`, exact unbind via the
  division-algebra inverse)
- **rhythm** - a verb/noun alternation drive

### Why Kaggle is a good fit
- The bottleneck is **RAM + sparse linear algebra**, *not* GPU - so set Accelerator =
  **None (CPU)**. 30 GB RAM lets us push vocab to ~100k and corpora to hundreds of
  millions of words.
- Build = co-occurrence counting -> shifted PPMI -> truncated SVD + n-gram tables. Minutes,
  not hours. The 12-hour limit is irrelevant.

### Scaling facts (measured on a 15 GB box, base64-verified)
- 46.5M-word PubMed corpus, **vocab 50k**: PPMI built in 39 s, full model in 153 s, **peak
  RAM 7.5 GB**. So Kaggle's 30 GB comfortably allows **vocab 100k+** and a much larger corpus.
- The one engineering fix that unlocks scale: accumulate the sparse co-occurrence
  **offset-by-offset into a CSR** (the naive single-COO build OOMs on the index arrays).

## 1. Get the code
Clone the repo (contains `octonion_lm.py` = the algebra, and `octonion_gpt.py` = the model).

In [ ]:
!git clone -q -b claude/octonionic-compression-tokenizer-nxbMq https://github.com/karahuseyn/flocking_birds.git
%cd flocking_birds
!pip -q install scipy numpy

## 2. Choose a large corpus
On Kaggle the HuggingFace / Wikimedia downloads that are blocked elsewhere work fine. Pick
ONE of these (or attach a Kaggle Dataset and point `CORPUS_PATH` at it).

- **WikiText-103** (~500 MB raw English, clean) - good general-language corpus.
- **C4 / OpenWebText shards** - web text, very large; take a few shards.
- **PubMed / arXiv** - domain text (the repo already builds a 335 MB PubMed corpus).

Below: WikiText-103 via the `datasets` library (streams, no full download needed).

In [ ]:
import os
CORPUS_PATH = '/kaggle/working/corpus_big.txt'
TARGET_CHARS = 400_000_000   # ~70M words; raise toward 1-2 GB on Kaggle's 30 GB RAM

if not os.path.exists(CORPUS_PATH):
    !pip -q install datasets
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='train', streaming=True)
    n = 0
    with open(CORPUS_PATH, 'w', encoding='utf-8') as f:
        for row in ds:
            t = row['text']
            if t and not t.startswith(' ='):     # skip heading lines
                f.write(t); n += len(t)
                if n >= TARGET_CHARS:
                    break
    print(f'wrote {n/1e6:.0f} MB to {CORPUS_PATH}')
else:
    print('corpus already present')

## 3. Build the gradient-free model (CPU, minutes)
Vocab 80k here - raise to 100k+ if RAM allows. The model is cached to `/kaggle/working`
so a re-run loads instantly.

In [ ]:
import octonion_gpt as G, time
VOCAB = 80_000
t0 = time.time()
text = open(CORPUS_PATH, encoding='utf-8').read()
M = G.build(text, vocab_size=VOCAB)          # PPMI -> sparse -> truncated SVD + n-grams
G.save(M, '/kaggle/working/octogpt_wikitext_%d' % VOCAB)
print('total build %.0f s' % (time.time() - t0))

## 4. Inspect the semantics and generate
Real gradient-free word geometry, then prompt-faithful generation.

In [ ]:
import numpy as np
emb, wi, vocab = M['emb'], M['wi'], M['vocab']
def neighbours(w, k=8):
    return [vocab[i] for i in (emb @ emb[wi[w]]).argsort()[::-1][1:k+1]] if w in wi else ['<oov>']
for w in ['king', 'science', 'war', 'music', 'love']:
    print(f'{w:9s}-> {neighbours(w)}')

In [ ]:
for prompt in ['the history of', 'in the year', 'the city was', 'she looked at']:
    print('\n' + prompt + ' |||')
    print('  ' + G.generate(M, prompt, n=60))

## 5. Tune (all gradient-free knobs)
`generate(M, prompt, temp=, drift=, w_subj=, w_pred=, w_alt=, rep_pen=, n=)`
- `temp` up -> more diverse, `rep_pen` up -> less repetition
- `drift` up -> topic moves faster (more argument-like), down -> stays on prompt
- `w_subj` / `w_pred` -> proposition (subject/predicate) binding strength
- `w_alt` -> subject/verb syntactic rhythm

Everything is O(n) per token and uses no gradients. Larger corpus + larger vocab is the
main lever for quality. See the repo README for the full module-by-module story.

## 6. Logic-guided reasoning (octonion_infer + octonion_qa)

Beyond fluent text, the octonion algebra can do **multi-hop logical inference** - the
procedural reasoning a pure n-gram (and arithmetic) cannot. An implication `A=>B` is an
exact octonion rotation, so chaining rules composes losslessly. `octonion_qa` parses
one-step rules from a prompt, then *derives* transitive answers it was never told.

In [ ]:
import octonion_qa as QA
qa = QA.OctonionQA()
qa.read('''
fever causes infection. infection causes inflammation.
inflammation causes tissuedamage. tissuedamage causes pain.
fever causes fatigue. fatigue causes dehydration.
''')
for q in ['does fever lead to pain?',
          'does infection lead to pain?',
          'does fever lead to dehydration?',
          'does pain lead to fever?',
          'does fever lead to coma?']:
    print(q, '->', qa.answer(q))

Expected: it derives `fever -> ... -> pain` (4 hops, never stated), refuses the wrong
direction (`pain -> fever`), and says *I don't know* for an unknown concept. This is
**derivation, not retrieval** - all gradient-free, exact octonion modus-ponens search.
Run `python3 octonion_infer.py` for the raw multi-hop proof engine.

## 7. Generation note: TTC-as-veto is on by default

`generate(..., veto=True)` spends light test-time compute on *rejecting* loop-forming
continuations (not maximizing a score, which degenerates into repetition). This keeps
output loop-free at no coherence cost - see KAGGLE.md for the measured comparison.